In [ ]:
!pip install --upgrade xgboost

In [ ]:
!pip install ydata-profiling

In [ ]:
!pip install graphviz

In [ ]:
!pip install pydotplus

In [ ]:
import zipfile
import numpy as np
import pandas as pd
from io import StringIO
from ydata_profiling import ProfileReport
from sklearn.utils import resample
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, roc_auc_score
from xgboost import XGBClassifier
from sklearn.tree import DecisionTreeClassifier, export_graphviz, export_text
from sklearn import tree
from sklearn.model_selection import train_test_split
from sklearn import metrics
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import pydotplus
from IPython.display import Image
from xgboost.plotting import to_graphviz
import graphviz

In [ ]:
with zipfile.ZipFile("/content/ddos.zip", 'r') as zip_ref:
    zip_ref.extractall("CSV DDos")  # Pasta de destino


In [ ]:
df = pd.read_csv("/content/CSV DDos/Portmap.csv")

df.head(5)


In [ ]:
# Remover espaços extras nos nomes das colunas
df.columns = df.columns.str.strip()

# Verifique novamente
print(df.columns.tolist())

In [ ]:
# Configuração leve do Pandas Profiling
profile = ProfileReport(
    df,
    minimal=True,
    correlations={"auto": {"calculate": True}},
    #missing_diagrams={"heatmap": False}
)

# Salva em arquivo (evita overload no notebook)
profile.to_file("relatorio.html")

In [ ]:
# Colunas que serão movidas para o novo DataFrame
atributos_removidos = ['Active Mean', 'Active Min', 'Idle Mean', 'Idle Min',
'Idle Std', 'Avg Fwd Segment Size', 'Average Packet Size', 'Avg Fwd Segment Size',
'Fwd Packet Length Max', 'Fwd Packet Length Mean', 'Fwd Packet Length Min',
'Min Packet Length', 'Packet Length Mean', 'Subflow Fwd Bytes', 'Total Length of Fwd Packets',
'Max Packet Length', 'Packet Length Variance', 'Packet Length Std',
'Init_Win_bytes_forward', 'Fwd PSH Flags', 'Avg Bwd Segment Size',
'Bwd Header Length', 'Bwd IAT Max', 'Bwd IAT Mean', 'Bwd IAT Min',
'Bwd IAT Total', 'Bwd Packet Length Max', 'Bwd Packet Length Mean',
'Bwd Packets/s', 'Down/Up Ratio', 'Flow IAT Std', 'Subflow Bwd Bytes',
'Total Backward Packets', 'Total Length of Bwd Packets', 'Fwd Header Length',
'Fwd Header Length.1', 'Bwd PSH Flags', 'Fwd URG Flags', 'Bwd URG Flags', 'FIN Flag Count',
'SYN Flag Count', 'RST Flag Count', 'PSH Flag Count', 'ACK Flag Count', 'URG Flag Count',
'CWE Flag Count', 'ECE Flag Count', 'Fwd Avg Bytes/Bulk', 'Fwd Avg Packets/Bulk',
'Fwd Avg Bulk Rate', 'Bwd Avg Bytes/Bulk', 'Bwd Avg Packets/Bulk', 'Bwd Avg Bulk Rate',
'Flow IAT Max', 'Flow IAT Mean']

# Criar novo DataFrame
df_removidos = df[atributos_removidos].copy()

# Remover essas colunas do DataFrame original
df.drop(columns=atributos_removidos, inplace=True)

In [ ]:
# Verificar resultados
df.head()

In [ ]:
df_removidos.head()

In [ ]:
# Configuração leve do Pandas Profiling
profile = ProfileReport(
    df,
    minimal=True,
    correlations={"auto": {"calculate": True}},
)

# Salva em arquivo (evita overload no notebook)
profile.to_file("relatorio_depois_de_removidos.html")

In [ ]:
# Remover espaços extras nos nomes das colunas
df.columns = df.columns.str.strip()

# Verifique novamente
print(df.columns.tolist())

In [ ]:
colunas_numericas = df.select_dtypes(include=['int64', 'float64']).columns.tolist()
for coluna in colunas_numericas:
  print(f"\n--- Estatísticas para '{coluna}' ---")
  print(f"Média: {df[coluna].mean():.2f}")
  print(f"Mediana: {df[coluna].median():.2f}")
  print(f"Desvio Padrão: {df[coluna].std():.2f}")
  print(f"Mínimo: {df[coluna].min()}")
  print(f"Máximo: {df[coluna].max()}")

In [ ]:
print("Valores ausentes por coluna:")
print(df.isnull().sum())

In [ ]:
# Substituir infinitos por NaN
df.replace([np.inf, -np.inf], np.nan, inplace=True)

# Remover linhas com valores NaN
df.dropna(inplace=True)

In [ ]:
print(df.isnull().sum())
print(df.shape)

In [ ]:
# Histograma geral
df[colunas_numericas].hist(bins=20, figsize=(20, 15))
plt.suptitle("Distribuição das variáveis numéricas")
plt.show()

# Mapa de calor das correlações
plt.figure(figsize=(12,10))
sns.heatmap(df[colunas_numericas].corr(), annot=True, fmt=".2f", cmap="coolwarm")
plt.title("Correlação entre variáveis numéricas")
plt.show()

In [ ]:
# Obter a matriz de correlação absoluta
corr_matrix = df[colunas_numericas].corr().abs()

# Encontrar os pares com maior correlação (excluindo auto-correlação)
high_corr = (corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
             .stack()
             .sort_values(ascending=False))

# Top 20 pares com maior correlação
print(high_corr.head(20))

In [ ]:
# Obter a matriz de correlação absoluta
corr_matrix = df[colunas_numericas].corr().abs()

# Criar máscara para selecionar apenas correlações abaixo de 0.5 (excluindo auto-correlação e duplicatas)
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))  # Triângulo superior
filtered_corr = corr_matrix.where(mask)  # Aplicar máscara

# Filtrar correlações entre 0 e 0.5 (excluindo auto-correlação = 1.0)
low_corr_features = (filtered_corr[(filtered_corr < 0.5) & (filtered_corr > 0)]
                   .stack()
                   .sort_values(ascending=False))

print("Pares de features com correlação entre 0 e 0.5:")
print(low_corr_features.head(20))

# Para selecionar as colunas que não têm nenhuma correlação > 0.8 com outras:
# 1. Encontrar todas as correlações > 0.5 (excluindo auto-correlação)
high_corr_pairs = (corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
                  .stack()
                  .where(lambda x: x > 0.5)
                  .dropna()
                  .index)

# 2. Listar todas as features envolvidas em correlações altas
high_corr_features = set()
for pair in high_corr_pairs:
    high_corr_features.add(pair[0])
    high_corr_features.add(pair[1])

# 3. Selecionar features que NÃO estão na lista de alta correlação
selected_features = [col for col in colunas_numericas if col not in high_corr_features]

print("\nFeatures selecionadas (sem correlações > 0.5 com outras):")
print(selected_features)

In [ ]:
# Obter a matriz de correlações menor que 0.5
selected_features = df[selected_features].corr().abs()

# Encontrar os pares com maior correlação (excluindo auto-correlação)
high_corr = (selected_features.where(np.triu(np.ones(selected_features.shape), k=1).astype(bool))
             .stack()
             .sort_values(ascending=False))

# Top 20 pares com maior correlação
print(high_corr.head(20))

In [ ]:
print(df['Label'].value_counts())
sns.countplot(x='Label', data=df)
plt.title("Distribuição das classes (normal x ataque)")
plt.show()

In [ ]:
# Separar classes
df_majority = df[df['Label'] == 'Portmap']
df_minority = df[df['Label'] == 'BENIGN']

# Reduzir a classe majoritária para ter o mesmo número de exemplos que a minoritária
df_majority_downsampled = resample(df_majority,
                                   replace=False,
                                   n_samples=len(df_minority),
                                   random_state=42)

# Juntar novamente
df_balanced = pd.concat([df_majority_downsampled, df_minority])

# Embaralhar o dataset final
df_balanced = df_balanced.sample(frac=1, random_state=42).reset_index(drop=True)


In [ ]:
print(df_balanced['Label'].value_counts())

sns.countplot(data=df_balanced, x='Label')
plt.title('Distribuição das Classes Após o Balanceamento')
plt.show()

In [ ]:
X = df_balanced.drop('Label', axis=1)  # Todas as colunas exceto o target
y = df_balanced['Label']  # Apenas a coluna de classificação

In [ ]:
# Primeira divisão: 60% treino, 40% temporário (val + teste)
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.4, random_state=42)

# Segunda divisão: 50% val, 50% teste (20% do total cada)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42)

# Verificar tamanhos
print("Treino:", X_train.shape, y_train.shape)
print("Validação:", X_val.shape, y_val.shape)
print("Teste:", X_test.shape, y_test.shape)

In [ ]:
print("\nDistribuição no Treino:")
print(y_train.value_counts(normalize=True))

print("\nDistribuição na Validação:")
print(y_val.value_counts(normalize=True))

print("\nDistribuição no Teste:")
print(y_test.value_counts(normalize=True))

In [ ]:
# Verificar e filtrar features existentes
print("Colunas disponíveis:", df_balanced.columns.tolist())

selected_features = [col for col in selected_features
                   if col in df_balanced.columns and col]

print("\nFeatures selecionadas após remoção:", selected_features)

# Codificar labels (0 e 1)
le = LabelEncoder()
df_balanced['Label'] = le.fit_transform(df_balanced['Label'])

# Separar features (X) e target (y)
X = df_balanced[selected_features]
y = df_balanced['Label']

# Dividir em treino (60%), validação (20%) e teste (20%)
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.4, random_state=42, stratify=y)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp)

# Padronizar os dados (evitar vazamento de dados)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

# Configurar modelo XGBoost com regularização
xgb_model = XGBClassifier(
    objective='binary:logistic',
    n_estimators=150,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.7,
    colsample_bytree=0.7,
    gamma=0.1,
    reg_alpha=0.1,
    reg_lambda=1.0,
    random_state=42,
    use_label_encoder=False,
    eval_metric=['logloss', 'error']
)

# Validação cruzada para verificar generalização
cv_scores = cross_val_score(xgb_model, X_train_scaled, y_train, cv=5, scoring='accuracy')
print(f"\nAcurácia média na validação cruzada: {cv_scores.mean():.4f} (±{cv_scores.std():.4f})")

# Treinamento com validação
try:
    xgb_model.fit(
        X_train_scaled, y_train,
        eval_set=[(X_val_scaled, y_val)],
        early_stopping_rounds=20,
        verbose=10
    )
except TypeError:
    print("\nEarly stopping não suportado - usando treinamento simples")
    xgb_model.fit(X_train_scaled, y_train)

def evaluate_model(model, X, y, set_name):
    y_pred = model.predict(X)
    y_proba = model.predict_proba(X)[:, 1]

    print(f"\n=== Avaliação ({set_name}) ===")
    print(f"Acurácia: {accuracy_score(y, y_pred):.4f}")
    print(f"AUC-ROC: {roc_auc_score(y, y_proba):.4f}")
    print("\nRelatório de Classificação:")
    print(classification_report(y, y_pred))

    # Matriz de confusão
    cm = confusion_matrix(y, y_pred)
    plt.figure(figsize=(6, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
    plt.title(f'Matriz de Confusão - {set_name}')
    plt.ylabel('Verdadeiro')
    plt.xlabel('Predito')
    plt.show()

# Avaliar em treino, validação e teste
evaluate_model(xgb_model, X_train_scaled, y_train, "Treino")
evaluate_model(xgb_model, X_val_scaled, y_val, "Validação")
evaluate_model(xgb_model, X_test_scaled, y_test, "Teste")

# Plot de importância das features
plt.figure(figsize=(10, 6))
sorted_idx = xgb_model.feature_importances_.argsort()
plt.barh(np.array(selected_features)[sorted_idx], xgb_model.feature_importances_[sorted_idx])
plt.title('Importância das Features (sem Flow Duration e Fwd Packets/s)')
plt.show()

# Exemplo de previsão
sample = X_test_scaled[0:1]
pred = xgb_model.predict(sample)
pred_proba = xgb_model.predict_proba(sample)

print("\nExemplo de previsão:")
print("Dados de exemplo (padronizados):", sample)
print(f"Classe prevista: {le.inverse_transform(pred)[0]}")
print("Probabilidades:", {le.classes_[0]: pred_proba[0][0].round(4),
                         le.classes_[1]: pred_proba[0][1].round(4)})

# Salvar modelo, scaler e encoder
artifacts = {
    'model': xgb_model,
    'scaler': scaler,
    'label_encoder': le,
    'features': selected_features
}

joblib.dump(artifacts, 'xgb_model_artifacts.pkl')
print("\nModelo e artefatos salvos com sucesso!")

In [ ]:
# Primeiro remova a coluna Flow ID
df_balanced = df_balanced.drop('Flow ID', axis=1)

# Depois separe em X e y como antes
X = df_balanced.drop('Label', axis=1)
y = df_balanced['Label']

In [ ]:
# Definir as features específicas que serão utilizadas
selected_features = [
    'Fwd IAT Std',
    'Fwd IAT Mean',
    'Init_Win_bytes_backward',
   # 'min_seg_size_forward',
   # 'Unnamed: 0',
    'Total Fwd Packets',
    'Fwd Packet Length Std',
    'Flow IAT Min',
    'Fwd IAT Total',
    'Fwd IAT Max',
    'Bwd IAT Std',
    'Subflow Fwd Packets',
   # 'act_data_pkt_fwd',
]

# Verificar e filtrar features existentes
print("Colunas disponíveis:", df_balanced.columns.tolist())
selected_features = [col for col in selected_features if col in df_balanced.columns]
print("\nFeatures selecionadas:", selected_features)

# Codificar labels (0 e 1)
le = LabelEncoder()
df_balanced['Label'] = le.fit_transform(df_balanced['Label'])

# Separar features (X) e target (y)
X = df_balanced[selected_features]
y = df_balanced['Label']

# Dividir em treino (60%), validação (20%) e teste (20%)
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.4, random_state=42, stratify=y)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp)

# Padronizar os dados (evitar vazamento de dados)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

# Configurar modelo XGBoost com regularização
xgb_model = XGBClassifier(
    objective='binary:logistic',
    n_estimators=150,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.7,
    colsample_bytree=0.7,
    gamma=0.1,
    reg_alpha=0.1,
    reg_lambda=1.0,
    random_state=42,
    use_label_encoder=False,
    eval_metric=['logloss', 'error']
)

# Validação cruzada para verificar generalização
cv_scores = cross_val_score(xgb_model, X_train_scaled, y_train, cv=5, scoring='accuracy')
print(f"\nAcurácia média na validação cruzada: {cv_scores.mean():.4f} (±{cv_scores.std():.4f})")

# Treinamento com validação
try:
    xgb_model.fit(
        X_train_scaled, y_train,
        eval_set=[(X_val_scaled, y_val)],
        early_stopping_rounds=20,
        verbose=10
    )
except TypeError:
    print("\nEarly stopping não suportado - usando treinamento simples")
    xgb_model.fit(X_train_scaled, y_train)

def evaluate_model(model, X, y, set_name):
    y_pred = model.predict(X)
    y_proba = model.predict_proba(X)[:, 1]

    print(f"\n=== Avaliação ({set_name}) ===")
    print(f"Acurácia: {accuracy_score(y, y_pred):.4f}")
    print(f"AUC-ROC: {roc_auc_score(y, y_proba):.4f}")
    print("\nRelatório de Classificação:")
    print(classification_report(y, y_pred))

    # Matriz de confusão
    cm = confusion_matrix(y, y_pred)
    plt.figure(figsize=(6, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
    plt.title(f'Matriz de Confusão - {set_name}')
    plt.ylabel('Verdadeiro')
    plt.xlabel('Predito')
    plt.show()

# Avaliar em treino, validação e teste
evaluate_model(xgb_model, X_train_scaled, y_train, "Treino")
evaluate_model(xgb_model, X_val_scaled, y_val, "Validação")
evaluate_model(xgb_model, X_test_scaled, y_test, "Teste")

# Plot de importância das features
plt.figure(figsize=(10, 6))
sorted_idx = xgb_model.feature_importances_.argsort()
plt.barh(np.array(selected_features)[sorted_idx], xgb_model.feature_importances_[sorted_idx])
plt.title('Importância das Features Selecionadas')
plt.show()

# Exemplo de previsão
sample = X_test_scaled[0:1]
pred = xgb_model.predict(sample)
pred_proba = xgb_model.predict_proba(sample)

print("\nExemplo de previsão:")
print("Dados de exemplo (padronizados):", sample)
print(f"Classe prevista: {le.inverse_transform(pred)[0]}")
print("Probabilidades:", {le.classes_[0]: pred_proba[0][0].round(4),
                         le.classes_[1]: pred_proba[0][1].round(4)})

# Salvar modelo, scaler e encoder
artifacts = {
    'model': xgb_model,
    'scaler': scaler,
    'label_encoder': le,
    'features': selected_features
}

joblib.dump(artifacts, 'xgb_model_artifacts.pkl')
print("\nModelo e artefatos salvos com sucesso!")

In [ ]:


# Criar e treinar um Decision Tree Classifier para visualização
dt_clf = DecisionTreeClassifier(
    max_depth=3,  # Limitar profundidade para melhor visualização
    random_state=42
)

# Treinar com as features selecionadas
dt_clf.fit(X_train_scaled, y_train)

# Avaliar a árvore de decisão
print("=== Avaliação da Árvore de Decisão ===")
y_pred_dt = dt_clf.predict(X_test_scaled)
print(f"Acurácia: {accuracy_score(y_test, y_pred_dt):.4f}")
print(classification_report(y_test, y_pred_dt))

# Visualização da árvore de decisão
dot_data = StringIO()

export_graphviz(dt_clf,
                out_file=dot_data,
                filled=True,
                rounded=True,
                special_characters=True,
                feature_names=selected_features,
                class_names=le.classes_.astype(str))  # Usar os nomes das classes reais

graph = pydotplus.graph_from_dot_data(dot_data.getvalue())
graph.write_png('decision_tree.png')
Image(graph.create_png())
  # Visualizar árvore específica do XGBoost com mais detalhes
try:

    # Visualizar a primeira árvore
    graphviz_source = to_graphviz(xgb_model, num_trees=0)
    graphviz_source.render(filename='xgboost_tree', format='png', cleanup=True)
    display(Image(filename='xgboost_tree.png'))

except ImportError:
    print("Graphviz não está disponível para visualização do XGBoost")

In [ ]:
# Visualização textual da árvore (alternativa)
plt.figure(figsize=(20, 10))
tree.plot_tree(dt_clf,
               feature_names=selected_features,
               class_names=le.classes_.astype(str),
               filled=True,
               rounded=True,
               fontsize=10)
plt.title('Árvore de Decisão - Visualização')
plt.show()

# Exportar regras de decisão
tree_rules = export_text(dt_clf, feature_names=selected_features)
print("=== Regras da Árvore de Decisão ===")
print(tree_rules)